<img src="assets/SIMEM_logo_1.png" width="1000" style="margin-bottom:8px"/>

[SIMEM – Sistema de Información Mercado Energía Mayorista Colombiano](https://www.simem.co/)


### Análisis de Variación de Demanda por Mercado de Comercialización

#### Capacitación Operadores de Red

**Pregunta de negocio:**  
Validar y analizar la variación de la demanda del OR, entender cómo se calculan las variaciones de demanda por Mercado de Comercialización, y determinar si el precio en bolsa presenta alguna relación con la demanda del mercado.

---

**Datasets utilizados:**
- **Demanda Comercial por MC** (`9b0967`): [SIMEM](https://www.simem.co/datadetail/9b096702-0070-429b-bcc5-f60f84b5dac9)
- **Precio de Bolsa** (`EC6945`): [SIMEM](https://www.simem.co/datadetail/EC694552-8887-481D-8152-8055B36DB8A1)


---
## Índice de contenidos

1. [Sección 1 – Parámetros y carga de datos](#Sección-1-–-Parámetros-y-carga-de-datos)
   - [1.1 Demanda Real por Mercado de Comercialización](#1.1-Demanda-Real-por-Mercado-de-Comercialización)
   - [1.2 Precio de Bolsa](#1.2-Precio-de-Bolsa)
2. [Sección 2 – Exploración inicial de la demanda](#Sección-2-–-Exploración-inicial-de-la-demanda)
3. [Sección 3 – Clasificación de tipos de día](#Sección-3-–-Clasificación-de-tipos-de-día)
4. [Sección 4 – Metodología de variación de demanda](#Sección-4-–-Metodología-de-variación-de-demanda)
5. [Sección 5 – Análisis](#Sección-5-–-Análisis)
   - [5.1 Variación ponderada de demanda](#5.1-Variación-ponderada-de-demanda)
   - [5.2 Variación ponderada – Regulado vs No Regulado (OR)](#5.2-Variación-ponderada-–-Regulado-vs-No-Regulado-(OR))
   - [5.3 Acumulado año 2026 vs 2025 – OR](#5.3-Acumulado-año-2026-vs-2025-–-OR)
6. [Sección 6 – Relación entre Precio de Bolsa y Demanda](#Sección-6-–-Relación-entre-Precio-de-Bolsa-y-Demanda)
7. [Resumen y conclusiones](#Resumen-y-conclusiones)

---

In [1]:
from pydataxm.pydatasimem import ReadSIMEM
import pandas as pd
import numpy as np
import os

from holidays_co import get_colombia_holidays_by_year

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "vscode"

---
## Sección 1 – Parámetros y carga de datos

Se ofrecen dos opciones para cargar los datos de demanda y precio de bolsa:
- **Opción CSV**: usar archivos locales ya descargados.
- **Opción SIMEM**: consultar directamente la API con `pydataxm`.


In [2]:
# ── Parámetros generales ────────────────────────────────────────────────────
FECHA_INICIO  = "2023-04-01"
FECHA_FIN     = "2026-04-30"

# IDs de datasets SIMEM
ID_DEMANDA_REAL = "9b0967"   # Demanda Real por MC
ID_PRECIO_BOLSA = "ec6945"   # Precio de Bolsa

# Ruta CSV de demanda (disponible localmente)
CSV_DEMANDA = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                           "DemandaRealPerdidasMC_2024-04-01_2026-04-30.csv")

# Mercado de comercialización a analizar en detalle
MC_ANALISIS = "CARIBE SOL"

# Activar fuente de datos: True = SIMEM API | False = CSV local
USAR_SIMEM_DEMANDA = True      # La demanda real se obtiene de SIMEM (si False, se lee desde CSV)
USAR_SIMEM_PRECIO  = True      # El precio solo está disponible en SIMEM


### 1.1 Demanda Real por Mercado de Comercialización

**Opción A – CSV local** (`USAR_SIMEM_DEMANDA = False`)  
**Opción B – API SIMEM** (`USAR_SIMEM_DEMANDA = True`)

Columnas del conjunto SIMEM: `FechaPublicacion`, `Fecha`, `CodigoSICAgente`, `TipoMercado`, `MercadoComercializacion`, `ClasificacionIndustrial`, `DdaReal`, `PerdidasEnergia`.

En el notebook se normalizan a nombres internos para el análisis: `FechaOperacion`, `codOperadorRed`, `CIIU`, `DemandaReal`, `Perdidas`.


In [3]:
if USAR_SIMEM_DEMANDA:
    # ── Opción B: consulta a la API de SIMEM ──────────────────────────────
    obj_dem = ReadSIMEM(dataset_id=ID_DEMANDA_REAL, start_date=FECHA_INICIO, end_date=FECHA_FIN)
    demanda = obj_dem.main()
    fuente_demanda = "SIMEM API"

    # Respaldo pragmático: algunos conjuntos pueden responder vacíos desde pydataxm
    if not isinstance(demanda, pd.DataFrame) or demanda.empty or len(demanda.columns) == 0:
        print("Advertencia: SIMEM no devolvió registros de demanda. Se usará el CSV local como respaldo.")
        demanda = pd.read_csv(CSV_DEMANDA)
        fuente_demanda = "CSV local (respaldo)"
else:
    # ── Opción A: CSV local ───────────────────────────────────────────────
    demanda = pd.read_csv(CSV_DEMANDA)
    fuente_demanda = "CSV local"

df_demanda = demanda.pivot_table(
    index=[
        "FechaPublicacion",
        "Fecha",
        "MercadoComercializacion",
        "CodigoSICAgente",
        "TipoMercado",
        "ClasificacionIndustrial",
    ],
    columns=["CodigoVariable"],
    values=["Valor"],
    aggfunc="sum",
).reset_index()

df_demanda.columns = [col[1] if col[1] else col[0] for col in df_demanda.columns.values]

# Normalizar nombres de columnas para que el resto del notebook use un esquema único
rename_map = {
    "Fecha": "FechaOperacion",
    "CodigoSICAgente": "codOperadorRed",
    "ClasificacionIndustrial": "CIIU",
    "DdaReal": "DemandaReal",
    "Demanda Real": "DemandaReal",
    "PerdidasEnergia": "Perdidas",
    "DemandaComercial": "DemandaComercial",
}
df_demanda = df_demanda.rename(columns={k: v for k, v in rename_map.items() if k in df_demanda.columns})

required_columns = [
    "FechaOperacion",
    "MercadoComercializacion",
    "codOperadorRed",
    "TipoMercado",
    "CIIU",
    "DemandaReal",
    "Perdidas",
    "DemandaComercial",
]
df_demanda["DemandaComercial"] = df_demanda["DemandaReal"] + df_demanda["Perdidas"]
missing_columns = [col for col in required_columns if col not in df_demanda.columns]
if missing_columns:
    raise KeyError(f"Faltan columnas requeridas tras normalizar la demanda: {missing_columns}. " f"Columnas disponibles: {list(df_demanda.columns)}")

df_demanda["FechaOperacion"] = pd.to_datetime(df_demanda["FechaOperacion"])

print(f"Fuente: {fuente_demanda}")
print(f"Registros: {len(df_demanda):,}")
print(f"Período  : {df_demanda['FechaOperacion'].min().date()} → {df_demanda['FechaOperacion'].max().date()}")
print(f"Mercados : {df_demanda['MercadoComercializacion'].nunique()}")
print(f"Columnas normalizadas: {required_columns}")
df_demanda.head(3)

****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Demanda Real y Perdidas por Operador de Red, Mercado de comercialización, Tipo de mercado y CIIU"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0030112266540527344
Extraccion de registros: 159.2166645526886
End of data extracting process
****************************************************************************************************
Fuente: SIMEM API
Registros: 395,957
Período  : 2023-04-01 → 2026-04-30
Mercados : 29
Columnas normalizadas: ['FechaOperacion', 'MercadoComercializacion', 'codOperadorRed', 'TipoMercado', 'CIIU', 'DemandaReal', 'Perdidas', 'DemandaComercial']


,FechaPublicacion,FechaOperacion,MercadoComercializacion,codOperadorRed,TipoMercado,CIIU,DemandaReal,Perdidas,DemandaComercial
0,2026-05-05,2026-04-01,ANTIOQUIA,EPMD,No Regulado,"ACTIVIDADES ARTÍSTICAS, DE ENTRETENEMIENTO Y R...",43538.95,705.85,44244.80
1,2026-05-05,2026-04-01,ANTIOQUIA,EPMD,No Regulado,ACTIVIDADES DE ATENCIÓN DE LA SALUD HUMANA Y D...,284459.42,4671.04,289130.46
2,2026-05-05,2026-04-01,ANTIOQUIA,EPMD,No Regulado,ACTIVIDADES DE LOS HOGARES INDIVIDUALES EN CAL...,3966.66,65.15,4031.81


### 1.2 Precio de Bolsa

**Opción C – CSV local** (si se dispone del archivo, cambiar `USAR_SIMEM_PRECIO = False` y ajustar `CSV_PRECIO`)  
**Opción D – API SIMEM** (`USAR_SIMEM_PRECIO = True`)


In [4]:
CSV_PRECIO = "precio_bolsa.csv"   # ajustar si se dispone del archivo local

if USAR_SIMEM_PRECIO:
    # ── Opción D: API SIMEM ───────────────────────────────────────────────
    obj_precio = ReadSIMEM(dataset_id=ID_PRECIO_BOLSA,
                           start_date=FECHA_INICIO, end_date=FECHA_FIN)
    df_precio_raw = obj_precio.main()
else:
    # ── Opción C: CSV local ───────────────────────────────────────────────
    df_precio_raw = pd.read_csv(CSV_PRECIO)

print(f"Registros precio: {len(df_precio_raw):,}")
print(f"Columnas: {list(df_precio_raw.columns)}")
df_precio_raw.head(3)

****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Precio de bolsa horario"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.008216381072998047
Extraccion de registros: 185.1984679698944
End of data extracting process
****************************************************************************************************
Registros precio: 528,096
Columnas: ['CodigoVariable', 'FechaHora', 'CodigoDuracion', 'UnidadMedida', 'Version', 'Valor']


,CodigoVariable,FechaHora,CodigoDuracion,UnidadMedida,Version,Valor
0,PB_Nal,2023-04-20T13:00:00,PT1H,COP/kWh,TXF,304.7428
1,PB_Tie,2023-04-20T03:00:00,PT1H,COP/kWh,TX1,270.0000
2,PB_Int,2023-04-20T15:00:00,PT1H,COP/kWh,TXR,333.8731


In [5]:
def filtrar_ultima_version(df, columna_fecha, columna_version, columna_variable=None):  # Función para filtrar la última versión de registros por mes y variables (opcional)
    fecha_mes_col = "año_mes"
    orden_version_col = "orden_version"
    versiones = ["TX1", "TX2", "TXR", "TXF", "TX3", "TX4", "TX5", "TX6", "TX7", "TX8", "TX9", "TX10"]  # Lista de versiones ordenadas por prioridad
    version_orden = pd.Series(range(1, len(versiones) + 1), index=versiones)  # Serie que asigna un orden numérico a cada versión
    df[columna_fecha] = pd.to_datetime(df[columna_fecha])  # Conversión de la columna de fecha a tipo datetime
    df[fecha_mes_col] = df[columna_fecha].dt.to_period("M")  # Se crea una columna con el año y mes del registro
    df[orden_version_col] = df[columna_version].map(version_orden)  # Se asigna el orden de versión a cada fila

    if columna_variable is None:  # Si no se especifican variables adicionales
        df_filtrado = df.sort_values([fecha_mes_col, orden_version_col], ascending=[True, False])  # Ordena por mes y versión descendente
        df_filtrado = df_filtrado.drop_duplicates(subset=[columna_fecha], keep="first")  # Elimina duplicados conservando la última versión por fecha
    else:  # Si se especifican variables adicionales
        if isinstance(columna_variable, str):  # Convierte a lista si es un solo string
            columna_variable = [columna_variable]
        orden_columnas = columna_variable + [fecha_mes_col, orden_version_col]  # Define el orden de columnas para ordenar
        df_filtrado = df.sort_values(orden_columnas, ascending=[True] * len(columna_variable) + [True, False])  # Ordena por variables, mes y versión
        df_filtrado = df_filtrado.drop_duplicates(subset=columna_variable + [columna_fecha], keep="first")  # Elimina duplicados por variables y fecha

    df_filtrado = df_filtrado.drop(columns=[fecha_mes_col, orden_version_col])  # Elimina columnas auxiliares
    return df_filtrado

In [6]:
df_precio_raw = filtrar_ultima_version(df=df_precio_raw, columna_fecha='FechaHora', columna_version='Version', columna_variable=['CodigoVariable'])

In [7]:
# Detectar columna de fecha y precio automáticamente
col_fecha_precio = next(
    (c for c in df_precio_raw.columns if "fecha" in c.lower() or "date" in c.lower()),
    df_precio_raw.columns[0]
)
col_precio_valor = next(
    (c for c in df_precio_raw.columns if "precio" in c.lower() or "price" in c.lower() or "bolsa" in c.lower()),
    df_precio_raw.columns[-1]
)

df_precio_raw[col_fecha_precio] = pd.to_datetime(df_precio_raw[col_fecha_precio])

# Agregar a precio diario promedio (el dataset suele ser horario)
df_precio_diario = (
    df_precio_raw
    .groupby(df_precio_raw[col_fecha_precio].dt.date)[col_precio_valor]
    .mean()
    .reset_index()
    .rename(columns={col_fecha_precio: "Fecha", col_precio_valor: "PrecioBolsa_COP_kWh"})
)
df_precio_diario["Fecha"] = pd.to_datetime(df_precio_diario["Fecha"])

print(f"Precio diario: {len(df_precio_diario)} días")
df_precio_diario.head(3)


Precio diario: 1126 días


,Fecha,PrecioBolsa_COP_kWh
0,2023-04-01,147.124333
1,2023-04-02,150.443500
2,2023-04-03,145.258275


---
## Sección 2 – Exploración inicial de la demanda

Antes de calcular variaciones, conviene conocer la estructura del dato:
- ¿Cuántos mercados hay y cuáles son los más grandes?
- ¿Cómo se distribuye entre mercado **Regulado** y **No Regulado**?


In [8]:
# Demanda total por MC en el período (suma de todo el período)
demanda_por_mc = (
    df_demanda
    .groupby("MercadoComercializacion")["DemandaComercial"]
    .sum()
    .div(1e6)                      # pasar a GWh (kWh → GWh: /1e6)
    .reset_index()
    .rename(columns={"DemandaComercial": "DemandaTotal_GWh"})
    .sort_values("DemandaTotal_GWh", ascending=False)
)

fig = px.bar(
    demanda_por_mc,
    x="MercadoComercializacion",
    y="DemandaTotal_GWh",
    title=f"Demanda total por Mercado de Comercialización ({FECHA_INICIO} – {FECHA_FIN})",
    labels={"MercadoComercializacion": "Mercado", "DemandaTotal_GWh": "Demanda [GWh]"},
    color="DemandaTotal_GWh",
    color_continuous_scale="Blues",
)
fig.update_layout(template="plotly_white", height=420,
                  xaxis_tickangle=-45, coloraxis_showscale=False)
fig.show()


In [9]:
# Composición Regulado vs No Regulado por MC (top 10 mercados)
top10 = demanda_por_mc.head(10)["MercadoComercializacion"].tolist()

demanda_tipo = (
    df_demanda[df_demanda["MercadoComercializacion"].isin(top10)]
    .groupby(["MercadoComercializacion", "TipoMercado"])["DemandaComercial"]
    .sum()
    .div(1e6)
    .reset_index()
    .rename(columns={"DemandaComercial": "DemandaTotal_GWh"})
    .sort_values("DemandaTotal_GWh", ascending=False)
)

fig = px.bar(
    demanda_tipo,
    x="MercadoComercializacion",
    y="DemandaTotal_GWh",
    color="TipoMercado",
    barmode="stack",
    title="Demanda Regulado vs No Regulado – Top 10 Mercados",
    labels={"MercadoComercializacion": "Mercado", "DemandaTotal_GWh": "Demanda [GWh]",
            "TipoMercado": "Tipo"},
)
fig.update_layout(template="plotly_white", height=400, xaxis_tickangle=-45)
fig.show()


---
## Sección 3 – Clasificación de tipos de día

Para calcular la variación de demanda de forma correcta, los días del mes se clasifican en tres tipos:

| Tipo | Descripción |
|------|-------------|
| **Comercial** | Lunes a viernes, sin ser festivo |
| **Sábado** | Sábados |
| **Domingo/Festivo** | Domingos o cualquier festivo nacional |

> La librería `holidays-co` provee los festivos oficiales de Colombia.


In [10]:
def obtener_festivos_colombia(años):
    """Retorna un conjunto de fechas (datetime.date) de festivos colombianos."""
    festivos = set()
    for año in años:
        festivos.update(holiday.date for holiday in get_colombia_holidays_by_year(año))
    return festivos


def clasificar_tipo_dia(fecha, festivos):
    """Clasifica una fecha en: Comercial, Sábado o Domingo/Festivo."""
    fecha_obj = fecha.date() if hasattr(fecha, "date") else fecha
    if fecha_obj.weekday() == 6 or fecha_obj in festivos:
        return "Domingo/Festivo"
    elif fecha_obj.weekday() == 5:
        return "Sábado"
    else:
        return "Comercial"


# Precargar festivos para todos los años del dataset
_años = df_demanda["FechaOperacion"].dt.year.unique().tolist()
FESTIVOS_CO = obtener_festivos_colombia(_años)

# Verificación rápida
print("Festivos Colombia cargados:", len(FESTIVOS_CO))
print("Ejemplo festivos 2025:", sorted([f for f in FESTIVOS_CO if f.year == 2025])[:5])


Festivos Colombia cargados: 71
Ejemplo festivos 2025: [datetime.date(2025, 1, 1), datetime.date(2025, 1, 6), datetime.date(2025, 3, 24), datetime.date(2025, 4, 17), datetime.date(2025, 4, 18)]


---
## Sección 4 – Metodología de variación de demanda

### ¿Por qué no se usa el simple mes actual / mes anterior?

El consumo de energía depende fuertemente de los tipos de día, en especial, del número de días comerciales del mes.  
Si en enero de un año hay 22 días comerciales y en el siguiente solo 21, la demanda total bajará aunque el consumo diario haya subido.

### Cálculo del crecimiento ponderado

El crecimiento mensual se calcula como el **promedio ponderado** de los crecimientos por tipo de día, usando como pesos el número de días de cada tipo **del mes actual**:

$$
C_{mes} = \frac{C_{com} \cdot N_{com} + C_{sab} \cdot N_{sab} + C_{dom/fest} \cdot N_{dom/fest}}{N_{total}}
$$

Donde el crecimiento por tipo de día compara el **promedio diario** del mes actual vs el mismo mes del año anterior:

$$
C_{tipo} = \left(\frac{\bar{D}_{actual,tipo}}{\bar{D}_{anterior,tipo}} - 1\right) \times 100\%
$$

**Ejemplo:** Si en enero 2026 (31 días) hay 21 días comerciales, 4 sábados y 6 domingos/festivos:

$$
C_{ene26} = \frac{5.87\% \times 21 + 5.23\% \times 4 + 5.26\% \times 6}{31} = 5.67\%
$$


In [11]:
def calcular_variacion_ponderada(df_mc, festivos, col_demanda="DemandaComercial",
                                  col_fecha="FechaOperacion"):
    """
    Calcula la variación ponderada mensual de demanda por tipo de día.

    Compara el mes actual con el mismo mes del año anterior.
    Retorna un DataFrame con la variación (%) y estadísticas auxiliares.
    """
    # Paso 1: agregar demanda diaria total del mercado
    dem_diaria = (
        df_mc.groupby(col_fecha)[col_demanda]
        .sum()
        .reset_index()
        .rename(columns={col_fecha: "Fecha", col_demanda: "DemandaDiaria"})
    )

    # Paso 2: clasificar tipo de día
    dem_diaria["TipoDia"] = dem_diaria["Fecha"].apply(
        lambda f: clasificar_tipo_dia(f, festivos)
    )
    dem_diaria["Año"] = dem_diaria["Fecha"].dt.year
    dem_diaria["Mes"] = dem_diaria["Fecha"].dt.month

    resultados = []
    TIPOS = ["Comercial", "Sábado", "Domingo/Festivo"]

    for (año, mes), g_actual in dem_diaria.groupby(["Año", "Mes"]):
        g_anterior = dem_diaria[
            (dem_diaria["Año"] == año - 1) & (dem_diaria["Mes"] == mes)
        ]
        if g_anterior.empty:
            continue

        total_dias = len(g_actual)
        variacion_pond = 0.0
        detalle = {}

        for tipo in TIPOS:
            dias_act = g_actual[g_actual["TipoDia"] == tipo]
            dias_ant = g_anterior[g_anterior["TipoDia"] == tipo]

            n = len(dias_act)
            if n == 0 or dias_ant.empty:
                detalle[tipo] = {"n_dias": n, "crec_pct": None,
                                 "prom_actual": None, "prom_anterior": None}
                continue

            prom_act = dias_act["DemandaDiaria"].mean()
            prom_ant = dias_ant["DemandaDiaria"].mean()

            if prom_ant == 0:
                continue

            crec = (prom_act / prom_ant - 1) * 100
            variacion_pond += crec * n
            detalle[tipo] = {"n_dias": n, "crec_pct": round(crec, 2),
                             "prom_actual": round(prom_act, 1),
                             "prom_anterior": round(prom_ant, 1)}

        variacion_pond /= total_dias

        resultados.append({
            "Año": año,
            "Mes": mes,
            "Periodo": f"{año}-{mes:02d}",
            "VariacionPonderada_pct": round(variacion_pond, 2),
            "DemandaMensual_GWh": round(g_actual["DemandaDiaria"].sum() / 1e6, 3),
            "N_Comerciales": detalle.get("Comercial", {}).get("n_dias", 0),
            "N_Sabados": detalle.get("Sábado", {}).get("n_dias", 0),
            "N_DomFest": detalle.get("Domingo/Festivo", {}).get("n_dias", 0),
            "Crec_Comercial_pct": detalle.get("Comercial", {}).get("crec_pct"),
            "Crec_Sabado_pct": detalle.get("Sábado", {}).get("crec_pct"),
            "Crec_DomFest_pct": detalle.get("Domingo/Festivo", {}).get("crec_pct"),
        })

    return pd.DataFrame(resultados)


print("Función 'calcular_variacion_ponderada' definida ✓")


Función 'calcular_variacion_ponderada' definida ✓


---
## Sección 5 – Análisis
### OR

En el mercado colombiano de energía:

- **OR (Operador de Red)**: gestiona la red de distribución en su zona.
- **Mercado Regulado**: usuarios residenciales y pequeñas empresas atendidos por el comercializador integrado al OR.
- **Mercado No Regulado**: grandes usuarios (≥ 0.1 MW) que eligieron libremente su comercializador.

La demanda **total del OR** = Regulado + No Regulado, pues toda la energía fluye por su red.


In [12]:
# Filtrar datos del OR
df_OR = df_demanda[df_demanda["MercadoComercializacion"] == MC_ANALISIS].copy()

print(f"Registros OR: {len(df_OR):,}")
print(f"OR   : {df_OR['codOperadorRed'].unique()}")
print(f"Tipos de mercado: {df_OR['TipoMercado'].unique()}")

# Demanda mensual desagregada por tipo
dem_OR_mensual = (
    df_OR
    .assign(
        AñoMes=df_OR["FechaOperacion"].dt.to_period("M").astype(str)
    )
    .groupby(["AñoMes", "TipoMercado"])["DemandaComercial"]
    .sum()
    .div(1e6)
    .reset_index()
    .rename(columns={"DemandaComercial": "DemandaTotal_GWh"})
)

fig = px.bar(
    dem_OR_mensual,
    x="AñoMes",
    y="DemandaTotal_GWh",
    color="TipoMercado",
    barmode="stack",
    title=f"Demanda mensual {MC_ANALISIS} – Regulado vs No Regulado",
    labels={"AñoMes": "Mes", "DemandaTotal_GWh": "Demanda [GWh]", "TipoMercado": "Tipo"},
)
fig.update_layout(template="plotly_white", height=400, xaxis_tickangle=-45)
fig.show()


Registros OR: 23,646
OR   : <ArrowStringArray>
['CSID', 'CSSD']
Length: 2, dtype: str
Tipos de mercado: <ArrowStringArray>
['No Regulado', 'Regulado']
Length: 2, dtype: str


### 5.1 Variación ponderada de demanda

Se aplica la metodología de variación ponderada por tipo de día al **total** del mercado (Regulado + No Regulado).  
Esto representa la variación de la demanda que el OR debe atender en su red.


In [13]:
# Variación total OR (OR = toda la red)
var_OR_total = calcular_variacion_ponderada(df_OR, FESTIVOS_CO)

print(f"Meses con comparación disponible: {len(var_OR_total)}")
print(f"\nTabla de variación ponderada – {MC_ANALISIS} Total (OR)")
print(var_OR_total[[
    "Periodo", "DemandaMensual_GWh", "VariacionPonderada_pct",
    "N_Comerciales", "N_Sabados", "N_DomFest",
    "Crec_Comercial_pct", "Crec_Sabado_pct", "Crec_DomFest_pct"
]].to_string(index=False))


Meses con comparación disponible: 25

Tabla de variación ponderada – CARIBE SOL Total (OR)
Periodo  DemandaMensual_GWh  VariacionPonderada_pct  N_Comerciales  N_Sabados  N_DomFest  Crec_Comercial_pct  Crec_Sabado_pct  Crec_DomFest_pct
2024-04             889.427                    9.83             22          4          4                8.90            11.47             13.35
2024-05             936.925                    4.30             21          4          6                2.86             4.50              9.23
2024-06             877.112                    0.04             18          5          7               -1.56             1.52              3.11
2024-07             945.807                    1.99             22          3          6                1.89            -1.28              3.99
2024-08             951.058                    1.67             20          5          6                1.45             1.83              2.26
2024-09             905.404                  

In [14]:
# Visualización de la variación ponderada mensual
fig = go.Figure()

fig.add_trace(go.Bar(
    x=var_OR_total["Periodo"],
    y=var_OR_total["VariacionPonderada_pct"],
    marker_color=["#d62728" if v < 0 else "#1f77b4"
                  for v in var_OR_total["VariacionPonderada_pct"]],
    name="Variación ponderada (%)",
    text=var_OR_total["VariacionPonderada_pct"].map(lambda v: f"{v:+.2f}%"),
    textposition="outside",
))

fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)

fig.update_layout(
    title=f"Variación ponderada de demanda – {MC_ANALISIS} (mismo mes año anterior)",
    xaxis_title="Período",
    yaxis_title="Variación (%)",
    template="plotly_white",
    height=400,
    showlegend=False,
)
fig.show()


### 5.2 Variación ponderada – Regulado vs No Regulado (OR)

Comparamos la variación de la demanda del **mercado regulado** (usuarios residenciales y pequeños comercios atendidos por CEOD) frente al **mercado no regulado** (grandes usuarios que eligen su comercializador).


In [15]:
var_regulado = calcular_variacion_ponderada(
    df_OR[df_OR["TipoMercado"] == "Regulado"], FESTIVOS_CO
).assign(Segmento="Regulado")

var_no_regulado = calcular_variacion_ponderada(
    df_OR[df_OR["TipoMercado"] == "No Regulado"], FESTIVOS_CO
).assign(Segmento="No Regulado")

var_comparacion = pd.concat([var_regulado, var_no_regulado], ignore_index=True)

fig = px.bar(
    var_comparacion,
    x="Periodo",
    y="VariacionPonderada_pct",
    color="Segmento",
    barmode="group",
    title=f"Variación ponderada de demanda – {MC_ANALISIS}: Regulado vs No Regulado",
    labels={"Periodo": "Período", "VariacionPonderada_pct": "Variación (%)", "Segmento": "Tipo"},
    text_auto=".2f",
    color_discrete_map={"Regulado": "#1f77b4", "No Regulado": "#ff7f0e"},
)
fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
fig.update_layout(template="plotly_white", height=420)
fig.show()


### 5.3 Acumulado año 2026 vs 2025 – OR

El crecimiento acumulado del año es la variación de la demanda total entre enero y el último mes disponible, con respecto al mismo período del año anterior.

$$
C_{acum} = \left(\frac{D_{ene-abr\,2026}}{D_{ene-abr\,2025}} - 1\right) \times 100\%
$$


In [16]:
def acumulado_año(df_mc, año_base, año_comp, meses_disp=None):
    """Calcula el crecimiento acumulado del año entre año_comp y año_base."""
    dem_mensual = (
        df_mc
        .assign(Año=df_mc["FechaOperacion"].dt.year,
                Mes=df_mc["FechaOperacion"].dt.month)
        .groupby(["Año", "Mes"])["DemandaComercial"]
        .sum()
        .div(1e6)
        .reset_index()
    )

    # Si no se especifican meses, usar los meses disponibles en año_comp
    if meses_disp is None:
        meses_disp = sorted(
            dem_mensual[dem_mensual["Año"] == año_comp]["Mes"].unique()
        )

    d_base = dem_mensual[
        (dem_mensual["Año"] == año_base) & (dem_mensual["Mes"].isin(meses_disp))
    ]["DemandaComercial"].sum()

    d_comp = dem_mensual[
        (dem_mensual["Año"] == año_comp) & (dem_mensual["Mes"].isin(meses_disp))
    ]["DemandaComercial"].sum()

    crec = (d_comp / d_base - 1) * 100 if d_base > 0 else None
    return d_base, d_comp, crec, meses_disp


d_2025, d_2026, crec_acum, meses = acumulado_año(df_OR, 2025, 2026)

print(f"Período de comparación: enero–{['','Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'][max(meses)]} ({len(meses)} meses)")
print(f"\n  Demanda 2025 acumulada : {d_2025:,.1f} GWh")
print(f"  Demanda 2026 acumulada : {d_2026:,.1f} GWh")
print(f"  Crecimiento acumulado  : {crec_acum:+.2f}%")


Período de comparación: enero–Abr (4 meses)

  Demanda 2025 acumulada : 3,563.7 GWh
  Demanda 2026 acumulada : 3,805.9 GWh
  Crecimiento acumulado  : +6.80%


---
## Sección 6 – Relación entre Precio de Bolsa y Demanda

**Hipótesis:** Los precios altos en bolsa pueden reflejar períodos de escasez hídrica (fenómeno El Niño) o baja oferta, que a su vez pueden correlacionarse con variaciones en la demanda (por ejemplo, por uso de autogeneración o respuesta de demanda de los grandes usuarios).

Se analiza la relación entre el **precio mensual promedio de bolsa** (COP/kWh) y la **demanda mensual total** del mercado.


In [17]:
# Precio mensual promedio
df_precio_mensual = (
    df_precio_diario
    .assign(AñoMes=df_precio_diario["Fecha"].dt.to_period("M").astype(str))
    .groupby("AñoMes")["PrecioBolsa_COP_kWh"]
    .mean()
    .reset_index()
    .rename(columns={"PrecioBolsa_COP_kWh": "PrecioPromedio_COP_kWh"})
)

# Demanda mensual OR total
dem_OR_mensual_total = (
    df_OR
    .assign(AñoMes=df_OR["FechaOperacion"].dt.to_period("M").astype(str))
    .groupby("AñoMes")["DemandaComercial"]
    .sum()
    .div(1e6)
    .reset_index()
    .rename(columns={"DemandaComercial": "DemandaTotal_GWh"})
)

# Cruzar las dos series
df_precio_demanda = dem_OR_mensual_total.merge(df_precio_mensual, on="AñoMes", how="inner")
print(f"Meses en común: {len(df_precio_demanda)}")
df_precio_demanda.head()


Meses en común: 37


,AñoMes,DemandaTotal_GWh,PrecioPromedio_COP_kWh
0,2023-04,800.894577,224.888742
1,2023-05,898.950575,568.232033
2,2023-06,881.023040,460.157961
3,2023-07,924.534120,520.454950
4,2023-08,936.069112,530.833034


In [18]:
# Evolución temporal: precio y demanda en dos ejes
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_precio_demanda["AñoMes"],
    y=df_precio_demanda["DemandaTotal_GWh"],
    mode="lines+markers",
    name="Demanda [GWh]",
    line=dict(color="#1f77b4", width=2),
    yaxis="y1",
))

fig.add_trace(go.Scatter(
    x=df_precio_demanda["AñoMes"],
    y=df_precio_demanda["PrecioPromedio_COP_kWh"],
    mode="lines+markers",
    name="Precio Bolsa [COP/kWh]",
    line=dict(color="#d62728", width=2, dash="dot"),
    yaxis="y2",
))

fig.update_layout(
    title=f"Demanda mensual vs Precio de Bolsa – {MC_ANALISIS}",
    xaxis=dict(title="Mes", tickangle=-45),
    yaxis=dict(title="Demanda [GWh]", title_font=dict(color="#1f77b4")),
    yaxis2=dict(title="Precio Bolsa [COP/kWh]", title_font=dict(color="#d62728"),
                overlaying="y", side="right"),
    template="plotly_white",
    height=420,
    legend=dict(x=0.01, y=0.99),
)
fig.show()


In [19]:
# Correlación y scatter
corr = df_precio_demanda[["DemandaTotal_GWh", "PrecioPromedio_COP_kWh"]].corr()
r = corr.loc["DemandaTotal_GWh", "PrecioPromedio_COP_kWh"]

fig = px.scatter(
    df_precio_demanda,
    x="PrecioPromedio_COP_kWh",
    y="DemandaTotal_GWh",
    text="AñoMes",
    trendline="ols",
    title=f"Correlación Precio de Bolsa vs Demanda – {MC_ANALISIS}  (r = {r:.3f})",
    labels={"PrecioPromedio_COP_kWh": "Precio Bolsa [COP/kWh]",
            "DemandaTotal_GWh": "Demanda [GWh]"},
)
fig.update_traces(textposition="top center", selector=dict(mode="markers+text"))
fig.update_layout(template="plotly_white", height=420)
fig.show()

print(f"\nCoeficiente de correlación de Pearson (r): {r:.4f}")
if abs(r) < 0.3:
    print("→ Correlación débil: el precio de bolsa NO explica significativamente la variación de la demanda en este mercado.")
elif abs(r) < 0.6:
    print("→ Correlación moderada: existe cierta relación entre precio y demanda.")
else:
    print("→ Correlación fuerte: el precio de bolsa y la demanda presentan alta co-variación.")



Coeficiente de correlación de Pearson (r): -0.3156
→ Correlación moderada: existe cierta relación entre precio y demanda.


---
## Resumen y conclusiones

| Aspecto | Hallazgo |
|---------|----------|
| **Metodología de variación** | El crecimiento ponderado por tipo de día elimina el sesgo por composición del calendario, dando una señal más estable que la simple comparación de totales mensuales. |
| **OR del Caribe Sol (CSSD)** | La demanda total (Regulado + No Regulado) refleja toda la energía que fluye por la red del OR. El mercado regulado domina en volumen. |
| **Mercado No Regulado** | Las variaciones del No Regulado pueden ser más volátiles, ya que responden a la actividad de sectores industriales concretos (minería, manufactura). |
| **Precio de bolsa** | Ver resultado de la correlación en la sección 6. En general, el precio de bolsa es una señal del lado de la **oferta** (hidrología, disponibilidad de plantas), mientras que la demanda responde principalmente a factores económicos y climáticos. |

> **Propuesta de valor:** monitorear las variaciones por MC permite anticipar necesidades de inversión en la red y coordinar con el CND las proyecciones de esta demanda.
